# Notebook 10.1  Speaker verification and a small diarizer, with honest per-condition evaluation

**Goal.** Extract speaker embeddings, score verification trials and compute the Equal Error Rate (EER) by sweeping the threshold, then run a simple clustering diarizer and compute the Diarization Error Rate (DER) frame by frame.

**How to use real data.** The cells run end to end on a small *synthetic* dataset so the notebook works with no downloads. Each step says how to swap in real audio and a real encoder: Arabic speaker material from [ArabCeleb](https://doi.org/10.1007/978-3-031-08421-8_23) or [SADA](https://doi.org/10.1109/ICASSP48485.2024.10446243), subject to their current licenses.

This notebook accompanies Chapter 10. The two metrics are implemented exactly as the algorithm boxes describe them: EER by threshold sweeping (Section 10.3) and DER by frame-by-frame accumulation (Section 10.4).

## 1. Setup

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
print('ready')

## 2. Speaker embeddings: real encoder or synthetic fallback

A speaker embedding is one fixed-length vector per utterance that captures the voice. In practice you extract it from a pretrained encoder such as ECAPA-TDNN. To keep this notebook self-contained, `embed` returns a synthetic embedding built from a per-speaker centroid plus utterance noise, so same-speaker pairs are close and different-speaker pairs are far.

**To use a real ECAPA-TDNN encoder** (SpeechBrain):
```python
from speechbrain.inference.speaker import EncoderClassifier
enc = EncoderClassifier.from_hparams(source='speechbrain/spkrec-ecapa-voxceleb')
import torchaudio
wav, sr = torchaudio.load(path)
emb = enc.encode_batch(wav).squeeze().detach().numpy()
```

In [ ]:
DIM = 64
def make_speakers(n_speakers=20, seed=1):
    g = np.random.default_rng(seed)
    return {f'spk{ i }': g.standard_normal(DIM) for i in range(n_speakers)}

def embed(speaker_centroid, spread=1.7):
    """Synthetic stand-in for a pooled ECAPA-TDNN embedding (length-normalized)."""
    v = speaker_centroid + spread*rng.standard_normal(DIM)
    return v / (np.linalg.norm(v) + 1e-9)

speakers = make_speakers()
print('speakers:', len(speakers), '| embedding dim:', DIM)

## 3. Verification trials and the Equal Error Rate

We build speaker-disjoint *trials*: genuine pairs (two utterances from the same speaker) and impostor pairs (two different speakers). Each trial gets a score, here cosine similarity between embeddings. Then we compute the EER by the four-step sweep of Section 10.3.

In [ ]:
def cosine(a, b):
    return float(np.dot(a, b) / ((np.linalg.norm(a)*np.linalg.norm(b)) + 1e-9))

def make_trials(speakers, n_genuine=400, n_impostor=400):
    names = list(speakers)
    genuine, impostor = [], []
    for _ in range(n_genuine):
        s = rng.choice(names)
        genuine.append(cosine(embed(speakers[s]), embed(speakers[s])))
    for _ in range(n_impostor):
        a, b = rng.choice(names, size=2, replace=False)
        impostor.append(cosine(embed(speakers[a]), embed(speakers[b])))
    return np.array(genuine), np.array(impostor)

genuine, impostor = make_trials(speakers)
print('genuine trials:', len(genuine), '| impostor trials:', len(impostor))
print('mean genuine score %.3f | mean impostor score %.3f' % (genuine.mean(), impostor.mean()))

In [ ]:
def compute_eer(genuine, impostor):
    """EER by threshold sweeping, exactly as in Section 10.3."""
    # 1. pool and sort all scores; each is a candidate threshold
    thresholds = np.sort(np.concatenate([genuine, impostor]))
    best = None
    for thr in thresholds:
        # 2. two rates at this threshold
        far = np.mean(impostor >= thr)   # impostors wrongly accepted
        frr = np.mean(genuine < thr)     # genuine wrongly rejected
        # 4. track where the two are closest
        gap = abs(far - frr)
        if best is None or gap < best[0]:
            best = (gap, thr, far, frr)
    gap, thr, far, frr = best
    eer = (far + frr) / 2
    return eer, thr

eer, thr = compute_eer(genuine, impostor)
print('EER = %.1f%%  at threshold %.3f' % (100*eer, thr))

In [ ]:
# plot FAR and FRR vs threshold; the crossing point is the EER
grid = np.linspace(min(impostor.min(), genuine.min()), max(impostor.max(), genuine.max()), 200)
far = [np.mean(impostor >= t) for t in grid]
frr = [np.mean(genuine < t) for t in grid]
fig, ax = plt.subplots(figsize=(6.4,4.2))
ax.plot(grid, np.array(far)*100, label='False Accept Rate')
ax.plot(grid, np.array(frr)*100, '--', label='False Reject Rate')
ax.axvline(thr, color='grey', ls=':', label='EER threshold')
ax.set_xlabel('threshold'); ax.set_ylabel('error rate (%)')
ax.set_title('FAR and FRR cross at the Equal Error Rate'); ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout(); fig.savefig('eer_curve.png', dpi=120)
print('saved eer_curve.png')

### Operating away from the EER

For authentication a false accept is costlier than a false reject, so a bank would raise the threshold to push the False Accept Rate very low, accepting more false rejects.

In [ ]:
strict = np.quantile(impostor, 0.99)   # almost no impostor scores exceed this
print('at a strict threshold %.3f:' % strict)
print('  FAR = %.2f%%  (impostors let in)' % (100*np.mean(impostor >= strict)))
print('  FRR = %.2f%%  (genuine users asked to retry)' % (100*np.mean(genuine < strict)))

## 4. A small diarizer and the Diarization Error Rate

Diarization answers who spoke when. We simulate a short two-speaker recording as a reference timeline, build a hypothesis (segment, embed, cluster), and score it with DER computed frame by frame, exactly as in Section 10.4.

In [ ]:
# reference timeline: list of (start, end, speaker) in seconds; includes a silence gap and one overlap
reference = [(0.0, 2.0,'A'), (2.0, 2.5,None), (2.5, 4.5,'B'), (4.5, 6.0,'A'), (5.5, 6.0,'B')]
DUR = 6.0

def timeline_at(segments, t):
    """set of speakers active at time t (None entries = silence)"""
    return {spk for (s,e,spk) in segments if spk is not None and s <= t < e}

# build a hypothesis by segmenting into 0.5s windows, embedding, and clustering
from sklearn.cluster import AgglomerativeClustering
win = 0.5
centroidA, centroidB = speakers['spk0'], speakers['spk1']
seg_times, seg_embs = [], []
for i in range(int(DUR/win)):
    t0 = i*win; tc = t0 + win/2
    active = timeline_at(reference, tc)
    if not active:
        continue   # a VAD would drop this silent window
    spk = sorted(active)[0]           # take the dominant speaker for this toy window
    cen = centroidA if spk=='A' else centroidB
    seg_times.append((t0, t0+win)); seg_embs.append(embed(cen))
labels = AgglomerativeClustering(n_clusters=2).fit_predict(np.array(seg_embs))
hypothesis = [(s, e, 'H%d'%lab) for (s,e),lab in zip(seg_times, labels)]
print('hypothesis segments:', len(hypothesis))

In [ ]:
def der(reference, hypothesis, dur, frame=0.01):
    """Frame-by-frame DER, as in Section 10.4. Returns DER and its three parts."""
    import itertools
    ref_spk = sorted({s for seg in reference for s in ([seg[2]] if seg[2] else [])})
    hyp_spk = sorted({seg[2] for seg in hypothesis})
    nframes = int(dur/frame)
    # try every mapping of hypothesis labels to reference speakers; keep the best
    best = None
    for perm in itertools.permutations(ref_spk, min(len(ref_spk), len(hyp_spk))):
        mapping = dict(zip(hyp_spk, perm))
        fa = miss = conf = total_ref = 0
        for k in range(nframes):
            t = k*frame
            R = timeline_at(reference, t)
            H = {mapping.get(s) for s in timeline_at(hypothesis, t)}
            H.discard(None)
            total_ref += len(R)
            # speaker confusion: matched count is the overlap
            correct = len(R & H)
            miss += max(0, len(R) - len(H))           # reference speakers not covered
            fa   += max(0, len(H) - len(R))           # extra hypothesized speakers
            conf += (min(len(R), len(H)) - correct)   # wrong-speaker assignments
        der_val = (fa + miss + conf) / (total_ref + 1e-9)
        if best is None or der_val < best[0]:
            best = (der_val, fa*frame, miss*frame, conf*frame)
    return best

der_val, fa_t, miss_t, conf_t = der(reference, hypothesis, DUR)
print('DER = %.1f%%' % (100*der_val))
print('  false alarm   %.2f s' % fa_t)
print('  missed speech %.2f s   (includes the undetected overlapped speaker)' % miss_t)
print('  confusion     %.2f s' % conf_t)

The undetected overlap (both A and B speak from 5.5 to 6.0 s, but the toy front end keeps one speaker per window) shows up as missed-speech time, exactly as Section 10.4 explains.

## 5. Where to go next

- Replace the synthetic embeddings with a real ECAPA-TDNN encoder (SpeechBrain) over ArabCeleb or SADA, and keep trials speaker-disjoint.
- Report EER and minimum detection cost per dialect and per channel (telephone, broadcast, far-field), not as a single pooled number.
- Swap the toy clustering diarizer for pyannote.audio, and report DER with a stated collar and overlap-scoring choice.
- Add an anti-spoofing countermeasure before accepting a verification decision for high-value actions (Section 10.5).